# TakeMeter v2: Training and Evaluation
Trains DistilBERT on `labels_final.csv` across **3 seeds**, runs a **TF-IDF baseline** on the same splits.

**Runtime → Change runtime type → T4 GPU** before running.

In [1]:
!pip install -q transformers datasets accelerate scikit-learn

## 1. Settings

In [11]:
LABELS = ["rumor", "news", "opinion"]
SEEDS = [42, 7, 2026]
MODEL_NAME = "distilbert-base-uncased"
EPOCHS = 10
LEARNING_RATE = 5e-5
BATCH_SIZE = 16
MAX_LENGTH = 256

# Criteria targets (from criteria.md)
MIN_LABEL_F1 = 0.60        # 1. every label F1
MIN_WORST_BEST_RATIO = 0.70  # 2. worst label F1 / best label F1
MIN_BASELINE_LEAD = 0.10   # 4. macro F1 lead over TF-IDF
MAX_ACC_SPREAD = 0.05      # 5. accuracy max minus min across seeds

## 2. Load data

In [12]:
import html, json
import numpy as np, pandas as pd
from google.colab import files

uploaded = files.upload()          # choose labels_final.csv
df = pd.read_csv(list(uploaded.keys())[0])

# Fix leftover HTML entities like &#x27; from the scraper
df["text"] = df["text"].astype(str).map(html.unescape)

LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
assert set(df["label"]) <= set(LABELS), f"Unknown labels: {set(df['label']) - set(LABELS)}"
df["label_id"] = df["label"].map(LABEL2ID)

print(len(df), "posts")
print(df["label"].value_counts())
print(f"Largest label: {df['label'].value_counts(normalize=True).max():.0%}")
print("Duplicate texts:", df["text"].duplicated().sum())

Saving labels_final.csv to labels_final (1).csv
272 posts
label
news       140
opinion     89
rumor       43
Name: count, dtype: int64
Largest label: 51%
Duplicate texts: 0


## 3. Split, baseline, and fine-tuning functions

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def split(seed):
    # 70 / 15 / 15, stratified so each split keeps the label balance
    train, temp = train_test_split(df, test_size=0.30, random_state=seed, stratify=df["label_id"])
    val, test = train_test_split(temp, test_size=0.50, random_state=seed, stratify=temp["label_id"])
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

def scores(y_true, y_pred):
    per_label = f1_score(y_true, y_pred, labels=list(range(len(LABELS))), average=None, zero_division=0)
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "label_f1": {LABELS[i]: round(float(f), 4) for i, f in enumerate(per_label)},
    }

def run_baseline(train, test):
    # Word-count model: no understanding of language, just which words appear
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(vec.fit_transform(train["text"]), train["label_id"])
    return clf.predict(vec.transform(test["text"]))

def to_dataset(d):
    ds = Dataset.from_pandas(d[["text", "label_id"]].rename(columns={"label_id": "labels"}))
    return ds.map(lambda x: tokenizer(x["text"], truncation=True, max_length=MAX_LENGTH), batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
            "accuracy": accuracy_score(labels, preds)}

def run_finetune(train, val, test, seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=f"./model_seed{seed}", num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=32,
        learning_rate=LEARNING_RATE, weight_decay=0.01,
        warmup_steps=max(1, int(0.1 * EPOCHS * ((len(train) + BATCH_SIZE - 1) // BATCH_SIZE))),
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        logging_steps=10, report_to="none", seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=to_dataset(train),
                      eval_dataset=to_dataset(val), data_collator=DataCollatorWithPadding(tokenizer),
                      compute_metrics=compute_metrics)
    trainer.train()
    out = trainer.predict(to_dataset(test))
    probs = torch.softmax(torch.tensor(out.predictions), dim=-1).numpy()
    return probs.argmax(axis=1), probs, trainer

## 4. Run all 3 seeds

In [14]:
results, last = [], None
for seed in SEEDS:
    print(f"\n========== SEED {seed} ==========")
    train, val, test = split(seed)
    print(f"train {len(train)} / val {len(val)} / test {len(test)}")
    print("test label counts:", test["label"].value_counts().to_dict())

    y = test["label_id"].values
    bl = scores(y, run_baseline(train, test))
    preds, probs, trainer = run_finetune(train, val, test, seed)
    ft = scores(y, preds)

    print("Baseline :", bl)
    print("DistilBERT:", ft)
    print(classification_report(y, preds, target_names=LABELS, zero_division=0))
    results.append({"seed": seed, "baseline": bl, "finetuned": ft})
    last = (seed, test, y, preds, probs, trainer)


========== SEED 42 ==========
train 190 / val 41 / test 41
test label counts: {'news': 21, 'opinion': 13, 'rumor': 7}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/190 [00:00<?, ? examples/s]

Map:   0%|          | 0/41 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,1.072460,0.938369,0.225806,0.512195
2,0.914037,0.655794,0.567843,0.780488
3,0.634321,0.517930,0.587692,0.804878
4,0.388535,0.445056,0.706926,0.804878
5,0.144955,0.484027,0.715576,0.780488
6,0.067652,0.593146,0.677502,0.829268
7,0.032277,0.604878,0.753713,0.853659
8,0.020096,0.600625,0.748792,0.853659
9,0.016631,0.621737,0.748792,0.853659
10,0.012378,0.629989,0.748792,0.853659


... (42 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/41 [00:00<?, ? examples/s]

Baseline : {'accuracy': 0.7073, 'macro_f1': 0.6314, 'label_f1': {'rumor': 0.3636, 'news': 0.7179, 'opinion': 0.8125}}
DistilBERT: {'accuracy': 0.7561, 'macro_f1': 0.7012, 'label_f1': {'rumor': 0.4, 'news': 0.7805, 'opinion': 0.9231}}
              precision    recall  f1-score   support

       rumor       0.33      0.50      0.40         6
        news       0.80      0.76      0.78        21
     opinion       1.00      0.86      0.92        14

    accuracy                           0.76        41
   macro avg       0.71      0.71      0.70        41
weighted avg       0.80      0.76      0.77        41



## 5. Criteria check

In [25]:
def check(name, values, ok):
    status = "MET" if all(ok) else "MISSED"
    print(f"{name:<38} " + "  ".join(f"{v:>7}" for v in values) + f"   {status}")
    return status

print(f"{'Criterion':<38} " + "  ".join(f"{'seed '+str(s):>7}" for s in SEEDS) + "   Verdict")
verdicts = {}

worst = [min(r["finetuned"]["label_f1"].values()) for r in results]
verdicts[1] = check(f"1. Every label F1 >= {MIN_LABEL_F1}", [f"{w:.2f}" for w in worst],
                    [w >= MIN_LABEL_F1 for w in worst])

ratio = [min(r["finetuned"]["label_f1"].values()) / max(max(r["finetuned"]["label_f1"].values()), 1e-9) for r in results]
verdicts[2] = check(f"2. Worst/best label F1 >= {MIN_WORST_BEST_RATIO}", [f"{x:.2f}" for x in ratio],
                    [x >= MIN_WORST_BEST_RATIO for x in ratio])

print(f"{'3. Cold relabel agreement >= 15/20':<38} (check by hand)")

lead = [r["finetuned"]["macro_f1"] - r["baseline"]["macro_f1"] for r in results]
verdicts[4] = check(f"4. Macro F1 lead over TF-IDF >= {MIN_BASELINE_LEAD}", [f"{x:+.2f}" for x in lead],
                    [x >= MIN_BASELINE_LEAD for x in lead])

accs = [r["finetuned"]["accuracy"] for r in results]
spread = max(accs) - min(accs)
verdicts[5] = check(f"5. Accuracy spread <= {MAX_ACC_SPREAD}", [f"{a:.2f}" for a in accs],
                    [spread <= MAX_ACC_SPREAD])
print(f"   (accuracy spread = {spread:.3f})")

print("\nPer-label F1 by seed:")
print(pd.DataFrame({r["seed"]: r["finetuned"]["label_f1"] for r in results}).T)

Criterion                              seed 42   seed 7  seed 2026   Verdict
1. Every label F1 >= 0.6                  0.36     0.55     0.40   MISSED
2. Worst/best label F1 >= 0.7             0.39     0.59     0.43   MISSED
3. Cold relabel agreement >= 15/20     (check by hand)
4. Macro F1 lead over TF-IDF >= 0.1      -0.02    -0.00    +0.07   MISSED
5. Accuracy spread <= 0.05                0.78     0.85     0.76   MISSED
   (accuracy spread = 0.098)

Per-label F1 by seed:
       rumor    news  opinion
42    0.3636  0.8000   0.9231
7     0.5455  0.8889   0.9231
2026  0.4000  0.7805   0.9231


## 6. Confusion matrix and wrong predictions (last seed)

In [27]:
seed, test, y, preds, probs, trainer = last
cm = confusion_matrix(y, preds, labels=list(range(len(LABELS))))
print(f"Confusion matrix, seed {seed} (rows = true, columns = predicted)\n")
print("| true \\ predicted | " + " | ".join(LABELS) + " |")
print("|---|" + "---|" * len(LABELS))
for i, l in enumerate(LABELS):
    print(f"| **{l}** | " + " | ".join(str(v) for v in cm[i]) + " |")

wrong = np.where(preds != y)[0]
print(f"\nWrong predictions: {len(wrong)} / {len(y)}\n")
for i in wrong:
    print(f"TRUE {LABELS[y[i]]:<8} PRED {LABELS[preds[i]]:<8} conf {probs[i][preds[i]]:.2f} | {test.text[i][:150]}")

Confusion matrix, seed 2026 (rows = true, columns = predicted)

| true \ predicted | rumor | news | opinion |
|---|---|---|---|
| **rumor** | 3 | 3 | 0 |
| **news** | 5 | 16 | 0 |
| **opinion** | 1 | 1 | 12 |

Wrong predictions: 10 / 41

TRUE opinion  PRED news     conf 0.95 | MC’s MUSIC LOUNGE: Tyla’s Sophomore Era. The road to the release of Tyla’s sophomore album has been a long one, but it finally arrives on July 24 th .
TRUE rumor    PRED news     conf 0.68 | Dad-of-12 Nick Cannon Honors Late Son Zen on Fifth Heavenly Birthday With Emotional Tribute: 'Turning Our Pain Into Purpose'. Nick Cannon posted a hea
TRUE news     PRED rumor    conf 0.72 | Angelina Jolie Reveals Her Daughters Helped the Actress Rediscover Herself: 'It’s Kind of Reminding Me What I May Have Lost'. Angelina Jolie said her 
TRUE rumor    PRED news     conf 0.87 | All My Children' Actress Marcy Walker Battling Mysterious Health Issue. Soap star Marcy Walker -- best known as Liza Colby on "All My Children" is in

## 7. Save results and model

In [28]:
with open("results_three_seeds.json", "w") as f:
    json.dump({"device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "settings": {"model": MODEL_NAME, "epochs": EPOCHS, "lr": LEARNING_RATE,
                            "batch": BATCH_SIZE, "max_length": MAX_LENGTH, "seeds": SEEDS},
               "runs": results, "verdicts": verdicts}, f, indent=2)

trainer.save_model("takemeter_model")
tokenizer.save_pretrained("takemeter_model")
!zip -qr takemeter_model.zip takemeter_model
files.download("results_three_seeds.json")
files.download("takemeter_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>